In [1]:
# pp.ipynb - preprocess count data and save into formatted adata files.

In [2]:
import anndata as ad
import numpy as np
import os
import pandas as pd
import scanpy as sc

In [3]:
simu_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim/simu"

cs_fn = os.path.join(simu_dir, "3_cs/cs.counts.h5ad")
rs_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim/baf_afc/afc.counts.cell_anno.h5ad"

out_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/internal_consistency/simu_modules_cs-rs/pp"

In [4]:
normal_cell_type = "normal"
tumor_cell_type = "tumor"

# Load Data

In [5]:
os.makedirs(out_dir, exist_ok = True)

## `cs` data and `rs` data

`cs` data and `rs` data have the same source of cell barcodes

In [6]:
cs = ad.read_h5ad(cs_fn)
cs

AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell_type', 'cell'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

In [7]:
cs = cs[:, ~cs.var["feature"].duplicated(keep = False)].copy()
cs.X = cs.layers["A"] + cs.layers["B"] + cs.layers["U"]
cs.obs.index = cs.obs["cell"]
cs.var.index = cs.var["feature"]
cs

AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell_type', 'cell'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

In [8]:
rs = ad.read_h5ad(rs_fn)
rs

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

In [9]:
rs = rs[:, ~rs.var["feature"].duplicated(keep = False)].copy()
rs.X = rs.layers["A"] + rs.layers["B"] + rs.layers["U"]
rs.obs.index = rs.obs["cell"]
rs.var.index = rs.var["feature"]
rs

AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

In [10]:
bool_all_in = np.all(rs.obs["cell"].isin(cs.obs["cell"]))
print(bool_all_in)
assert bool_all_in == True

True


### Use the same order of cells

In [11]:
cs = cs[rs.obs.index, :].copy()
cs

AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell_type', 'cell'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

## Use intersect genes

In [12]:
adata_list = [cs, rs]

genes = None
for i, adata in enumerate(adata_list):
    if i == 0:
        genes = adata.var["feature"].to_numpy()
    else:
        genes = np.intersect1d(genes, adata.var["feature"])
genes.shape

(32295,)

### Use the same order of genes

In [13]:
cs = cs[:, cs.var["feature"].isin(genes)].copy()
print(cs)

rs = rs[:, cs.var.index].copy()
print(rs)

AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell_type', 'cell'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'


## Split by cell type

In [14]:
cs_normal = cs[cs.obs["cell_type"] == normal_cell_type, :].copy()
print(cs_normal)

cs_tumor = cs[cs.obs["cell_type"] == tumor_cell_type, :].copy()
print(cs_tumor)

rs_normal = rs[rs.obs["cell"].isin(cs_normal.obs["cell"]), :].copy()
print(rs_normal)

rs_tumor = rs[rs.obs["cell"].isin(cs_tumor.obs["cell"]), :].copy()
print(rs_tumor)

AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell_type', 'cell'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell_type', 'cell'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'


# Save Data

### Check the order of cells and genes

In [15]:
assert np.all(cs_normal.obs["cell"].to_numpy() == rs_normal.obs["cell"].to_numpy())
assert np.all(cs_tumor.obs["cell"].to_numpy() == rs_tumor.obs["cell"].to_numpy())

In [16]:
assert np.all(cs_tumor.var["feature"].to_numpy() == cs_normal.var["feature"].to_numpy())
assert np.all(rs_normal.var["feature"].to_numpy() == cs_normal.var["feature"].to_numpy())
assert np.all(rs_tumor.var["feature"].to_numpy() == cs_normal.var["feature"].to_numpy())

In [17]:
cs_normal.write_h5ad(os.path.join(out_dir, "cs_normal.h5ad"), compression = 'gzip')
cs_tumor.write_h5ad(os.path.join(out_dir, "cs_tumor.h5ad"), compression = 'gzip')
rs_normal.write_h5ad(os.path.join(out_dir, "rs_normal.h5ad"), compression = 'gzip')
rs_tumor.write_h5ad(os.path.join(out_dir, "rs_tumor.h5ad"), compression = 'gzip')